# 06 · ConvNeXt —— 用 Transformer 的经验重造 CNN（家族收官）

**家族位置**：`02_CNN_Family` 第 6 个项目（收官站）。ViT 同台对比为**预留延续点**——`06_Transformer_Vision_Multimodal` 家族完工后回填，协议已兼容。

**与上一站的衔接**：05 的 Acc-FLOPs 地图摆出了家族六配置的性价比。本章回答最后一个问题：**如果 2012 年之后 CNN 重来一次，带着 Transformer 时代的经验，长什么样？** 答案是 ConvNeXt（2022）——不改卷积的归纳偏置，只改工程细节，就能一路追平 Swin Transformer。我们沿原论文的"现代化路径"对 ResNet20 做**逐步改造**，每步只动一个开关。

**学习目标**
1. 现代化五件套各自的账：深度可分离 DW、大核 7×7、倒瓶颈、LayerNorm+GELU、patchify stem
2. 路径消融：ResNet20 → +DW → +倒瓶颈 → +大核 → ConvNeXt-mini，每步买到什么
3. 优化器现代化：AdamW 回调（03 的 Adam 教训在这里闭环）
4. ConvNeXt-mini 的极端性价比：参数 41 万但 MACs 全家最小（3.9M）

## 1. 原理：给老房子做现代化翻新

### 通俗理解

**一句话**：ConvNeXt 不换承重墙（卷积的局部性/权值共享还在），只做装修——把 Transformer 时代验证过的好零件一件件换进 ResNet：更大的感受野（7×7 DW）、更顺的归一化（LayerNorm）、更平滑的激活（GELU）、更合理的算力分布（倒瓶颈）和直通_token 的入口（patchify stem）。

**比喻**：老式相机（ResNet）换上现代镜头（大核 DW）、电子取景器（LN）、更顺的快门曲线（GELU），外壳不动——性能直逼数码相机（Transformer），但成本结构完全不同。

### 现代化五件套（本项目的消融路径）

| 步骤 | 改动 | 原论文动机 |
|---|---|---|
| 锚点 | ResNet20（3×3 稠密卷积） | 出发点 |
| ① | 3×3 稠密 → **DW 3×3**（ResNeXt-ify） | 卷积各通道独立处理，算力省给通道混合 |
| ② | → **倒瓶颈**（1×1 扩张×4 → DW → 1×1） | 中间层在低分辨率下做重计算（MobileNetV2 方向） |
| ③ | DW 3×3 → **DW 7×7**（大核） | 与 Transformer 的全局注意力对齐的感受野 |
| ④ | + **LN / GELU / patchify 4×4 / 分离下采样 / γ 缩放** | 微观全面现代化 → ConvNeXt-mini |
| ⑤ | 优化器 SGD → **AdamW(1e-3, wd=0.05)** | 现代配方（03 的 Adam 教训闭环：BN 深网配错 lr 会翻车，现代组件配 AdamW 是正解） |

### 算力分布的账

倒瓶颈 + patchify 把计算从高分辨率挤到低分辨率：ConvNeXt-mini 41 万参数但只有 **3.9M MACs**（ResNet20 的 1/10）——参数多、计算少，与 DenseNet（参数少计算多）正好相反。

In [ ]:
import sys
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import CIFAR10_CLASSES, load_cifar10_torch
from common.engine import fit
from common.models import ConvNeXtCIFAR, ModernPathCIFAR, ResNetCIFAR
from common.utils import count_flops, count_params, set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE, "| torch:", torch.__version__)

## 2. 数据：CIFAR-10（同 03-05）

同一份数据、同一子集划分（前 10k）、同一标准化——家族锚点链的最后一站。

In [ ]:
Xtr, ytr, Xte, yte = load_cifar10_torch(str(ROOT / "data"))
print("训练集:", Xtr.shape, "| 测试集:", Xte.shape)

fig, axes = plt.subplots(2, 10, figsize=(12, 3.0))
for r in range(2):
    for c in range(10):
        idx = int(np.where(ytr.numpy() == c)[0][r])
        img = Xtr[idx].permute(1, 2, 0).numpy()
        img = (img * [0.2470, 0.2435, 0.2616] + [0.4914, 0.4822, 0.4465]).clip(0, 1)
        axes[r, c].imshow(img)
        axes[r, c].set_title(CIFAR10_CLASSES[c], fontsize=8)
        axes[r, c].axis("off")
plt.suptitle("CIFAR-10：每类两个样本", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig0_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. 模型定稿：六臂全家福

路径驱动器 `ModernPathCIFAR` 保持 ResNet20 骨架（stem/三阶段 16→32→64/下采样/GAP 完全一致），只换块内部；ConvNeXt-mini 则全套现代化（patchify 4×4 直接把 32×32 压到 8×8，dims 32→64→128，每阶段 2 块）。

In [ ]:
ARMS = {
    "ResNet20(锚点)":     (lambda: ResNetCIFAR(), dict(optimizer="sgd")),
    "①+DW 3×3":          (lambda: ModernPathCIFAR(path="dw"), dict(optimizer="sgd")),
    "②+倒瓶颈":           (lambda: ModernPathCIFAR(path="inv3"), dict(optimizer="sgd")),
    "③+大核 7×7":         (lambda: ModernPathCIFAR(path="inv7"), dict(optimizer="sgd")),
    "④ConvNeXt-mini":    (lambda: ConvNeXtCIFAR(), dict(optimizer="sgd")),
    "④+AdamW(现代配方)":   (lambda: ConvNeXtCIFAR(), dict(optimizer="adamw")),
}
print(f"{'臂':22s} {'参数量':>9s} {'MACs':>8s}")
for name, (cls, _) in ARMS.items():
    m = cls()
    print(f"{name:22s} {count_params(m):>9,} {count_flops(m)/1e6:>7.1f}M")

## 4. 主实验：现代化路径阶梯（约 23 分钟 CPU）

**协议**：10k 训练子集、**6 epochs**（③大核臂吞吐 92 img/s，为全梯同预算统一缩到 6ep）、SGD(momentum=0.9, lr=0.05) + wd=1e-4、batch 128、seed=0、测试全量 10k；⑤臂换 AdamW(1e-3, wd=0.05)。ResNet20 锚点在本协议下**同预算复训**，阶梯每一级严格可比。

In [ ]:
import time

EPOCHS = 6
tr = DataLoader(TensorDataset(Xtr[:10000], ytr[:10000]), batch_size=128, shuffle=True)
te = DataLoader(TensorDataset(Xte, yte), batch_size=512)

results, models = {}, {}
for name, (cls, cfg) in ARMS.items():
    set_seed(0)
    model = cls()
    kw = dict(epochs=EPOCHS, device=DEVICE, verbose=False)
    if cfg["optimizer"] == "sgd":
        kw.update(lr=0.05, optimizer_cls=partial(torch.optim.SGD, momentum=0.9), weight_decay=1e-4)
    else:
        kw.update(lr=1e-3, optimizer_cls=partial(torch.optim.AdamW), weight_decay=0.05)
    t0 = time.time()
    hist = fit(model, tr, te, **kw)
    results[name] = hist
    models[name] = model
    print(f"{name:22s} val_acc={hist['val_acc'][-1]:.2%} | val_loss={hist['val_loss'][-1]:.4f} | train_acc={hist['train_acc'][-1]:.2%} | {time.time()-t0:.0f}s", flush=True)

In [ ]:
colors = ["#AAAAAA", "#4C72B0", "#55A868", "#C44E52", "#8172B2", "#DD8452"]

fig, ax = plt.subplots(figsize=(9, 4.4))
for (name, _), c in zip(ARMS.items(), colors):
    ax.plot(results[name]["val_acc"], marker="o", ms=4, label=name, color=c)
ax.set_xlabel("epoch"); ax.set_ylabel("val_acc")
ax.set_title("现代化路径阶梯（10k 子集 · 6 epochs · seed=0）")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()

accs = {n: results[n]["val_acc"][-1] for n in ARMS}
params = {n: count_params(cls()) for n, (cls, _) in ARMS.items()}
macs = {n: count_flops(cls()) for n, (cls, _) in ARMS.items()}

names = list(ARMS)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
bars = axes[0].bar(range(len(names)), [accs[n] for n in names], color=colors)
for i, (b, n) in enumerate(zip(bars, names)):
    axes[0].text(b.get_x() + b.get_width() / 2, accs[n], f"{accs[n]:.2%}", ha="center", va="bottom", fontsize=8)
    axes[0].text(b.get_x() + b.get_width() / 2, 0.06, f"{params[n]/1e3:.0f}k\n{macs[n]/1e6:.0f}M",
                 ha="center", fontsize=7, color="white", fontweight="bold")
axes[0].set_xticks(range(len(names))); axes[0].set_xticklabels(names, rotation=20, ha="right", fontsize=8)
axes[0].set_ylim(0, 0.72); axes[0].set_ylabel("val_acc")
axes[0].set_title("现代化阶梯：每一步买到什么（柱底=参数/MACs）")
for n, c in zip(names, colors):
    axes[1].scatter(macs[n] / 1e6, accs[n], s=170, color=c, zorder=3)
    axes[1].annotate(n.split("(")[0], (macs[n] / 1e6, accs[n]), textcoords="offset points", xytext=(7, -12), fontsize=8)
axes[1].set_xlabel("MACs (百万/图)"); axes[1].set_ylabel("val_acc")
axes[1].set_title("现代化路径的 Acc-MACs 轨迹"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / "fig2_ladder.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n逐步增量（val_acc）:")
prev = None
for n in names:
    d = "" if prev is None else f"（{accs[n] - prev:+.2%}）"
    print(f"  {n:22s} {accs[n]:.2%} {d}")
    prev = accs[n]

## 5. 冠军的错误

冠军模型的混淆矩阵——现代化之后的错误结构是否改变（03 ResNet32：dog→cat 231；05 MobileNet：dog→cat 283）。

In [ ]:
best_name = max(accs, key=accs.get)
print("冠军:", best_name, f"{accs[best_name]:.2%}")
best_model = models[best_name]
best_model.eval()
with torch.no_grad():
    pred = best_model(Xte).argmax(1)

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(yte.numpy(), pred.numpy())
fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(10)); ax.set_yticklabels(CIFAR10_CLASSES, fontsize=8)
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=6)
ax.set_xlabel("预测"); ax.set_ylabel("真实"); ax.set_title(f"{best_name} 混淆矩阵")
plt.colorbar(im)
plt.tight_layout()
plt.savefig(FIGS / "fig3_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
flat = cm_off.ravel().argsort()[::-1][:5]
for k in flat:
    r, c = np.unravel_index(k, cm.shape)
    print(f"  {CIFAR10_CLASSES[r]:10s} → {CIFAR10_CLASSES[c]:10s} : {cm_off[r, c]} 次")

## 6. 总结与 ViT 预留延续点

**本项目收获**

1. 现代化五件套各自贡献的实测阶梯（结果见 §4）
2. ConvNeXt-mini 的极端性价比：参数 41 万 / MACs 3.9M——patchify+倒瓶颈把计算挤到低分辨率的账
3. 优化器闭环：ConvNeXt 组件 + AdamW 才是现代配方的完整形态（03 的 Adam 教训正面回收）
4. 家族收官：LeNet→ConvNeXt 十四年演进全部同协议复现，锚点链完整

**ViT 同台（预留）**：`06_Transformer_Vision_Multimodal` 家族完工后，用同协议（10k 子集 / 同配方的 AdamW / 同测试集）回填 ViT-mini 对比臂——本 notebook 的 `ARMS` 结构可直接扩展，Acc-MACs 地图加一个 ViT 点即可。CNN 与 ViT 的对比本质是"归纳偏置 vs 数据规模"的对照实验，届时家族证据链闭合。